# Day 5 - Code breaks, block 2: LSTM and GRU

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the value marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

## Setup - run these two cells first
The first cell loads PyTorch.

In [1]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

print("ready - PyTorch", torch.__version__)

ready - PyTorch 2.14.1+cu130


The second cell defines the **memory task** of code break 2. **Just run it - you do not need to read it.**
- `memory_batch(n, T)` - `n` random sequences of length `T`: at every step a random bit (+1 or -1), plus a marker
  at the first step. The label is the **first** bit - the network must remember it for `T` steps.
- `n_params(layer)` - counts the weights of a layer
- `train_memory(cell, T)` - trains a small network (`cell` = "RNN", "GRU" or "LSTM", 32 memory units) for 500 steps
  and returns the accuracy on 2 000 new sequences. 0.5 = guessing.

In [2]:
def memory_batch(n, T):
    x = torch.zeros(n, T, 2)
    x[:, :, 0] = torch.randint(0, 2, (n, T)) * 2.0 - 1      # a random bit at every time step
    x[:, 0, 1] = 1.0                                         # marker: "this is the first step"
    return x, (x[:, 0, 0] > 0).float()                       # label: was the FIRST bit +1?


class MemoryNet(nn.Module):
    def __init__(self, cell, hidden=32):
        super().__init__()
        if cell == "RNN":
            self.rnn = nn.RNN(2, hidden, batch_first=True)
        elif cell == "GRU":
            self.rnn = nn.GRU(2, hidden, batch_first=True)
        elif cell == "LSTM":
            self.rnn = nn.LSTM(2, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)
        if cell != "RNN":     # start the "keep" gate (LSTM forget gate, GRU update gate) in "remember" mode
            with torch.no_grad():
                self.rnn.bias_hh_l0[hidden:2 * hidden] = 3.0

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.out(out[:, -1]).squeeze(-1)             # many-to-one: the last memory


def n_params(layer):
    return sum(p.numel() for p in layer.parameters())


def train_memory(cell, T, steps=500):
    torch.manual_seed(0)
    net = MemoryNet(cell)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(steps):
        x, y = memory_batch(128, T)
        loss = F.binary_cross_entropy_with_logits(net(x), y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
    x, y = memory_batch(2000, T)
    with torch.no_grad():
        return ((net(x) > 0).float() == y).float().mean().item()


print("helpers ready")

helpers ready


## 💻 Code break 1 - RNN, GRU, LSTM: one word changes
8 sequences (for example heartbeats) of 140 time steps with 1 number per step. The three layers get the same
arguments - only the name changes.

In [17]:
HIDDEN = 16          # 👈 size of the memory

beats = torch.randn(8, 140, 1)                                   # (batch, time steps, features)
rnn = nn.RNN(input_size=1, hidden_size=HIDDEN, batch_first=True)
gru = nn.GRU(input_size=1, hidden_size=HIDDEN, batch_first=True)
lstm = nn.LSTM(input_size=1, hidden_size=HIDDEN, batch_first=True)

out, h = rnn(beats)
print("RNN   out:", tuple(out.shape), "  parameters:", n_params(rnn), "  last memory h:", tuple(h.shape))
out, h = gru(beats)
print("GRU   out:", tuple(out.shape), "  parameters:", n_params(gru), "  last memory h:", tuple(h.shape))
out, (h, c) = lstm(beats)                                        # the LSTM gives back TWO memories
print("LSTM  out:", tuple(out.shape), "  parameters:", n_params(lstm), "  last memory h:", tuple(h.shape),
      "  cell state c:", tuple(c.shape))

RNN   out: (8, 140, 16)   parameters: 304   last memory h: (1, 8, 16)
GRU   out: (8, 140, 16)   parameters: 912   last memory h: (1, 8, 16)
LSTM  out: (8, 140, 16)   parameters: 1216   last memory h: (1, 8, 16)   cell state c: (1, 8, 16)


🔧 **Change it:** `HIDDEN = 32`. Which numbers change?

<details><summary>👀 What you should see</summary>

- `out` is **(8, 140, 16)** for all three - the rest of your model does not change
- parameters: RNN **304**, GRU **912** (3 ×), LSTM **1 216** (4 ×) - a GRU has 3 small "RNNs" inside (2 gates +
  the candidate), an LSTM has 4 (3 gates + the candidate)
- the LSTM gives back **two** memories: h and the cell state c (the conveyor belt), both (1, 8, 16). That is why
  it needs `out, (h, c) = lstm(beats)`. If you only need `out`, write `out, _ = ...` - then the same line works
  for all three (we do that in the lab)
- HIDDEN = 32: out (8, 140, 32); RNN **1 120**, GRU **3 360**, LSTM **4 480**
</details>

## 💻 Code break 2 - Remember the first bit
The task from the slide: read `T` random bits, then answer: what was the **first** bit? Training takes a few seconds.

In [20]:
CELL = "RNN"         # 👈 "RNN", "GRU" or "LSTM"
T = 20               # 👈 10, 30, 60

start = time.time()
acc = train_memory(CELL, T)
print(f"{CELL}, T = {T}: test accuracy {acc:.2f}   (0.5 = guessing)   {time.time() - start:.0f} s")

RNN, T = 20: test accuracy 1.00   (0.5 = guessing)   4 s


🔧 **Change it:** fill in the table on the slide - `T = 10`, `30`, `60` for `RNN`, then for `LSTM` and `GRU`.

<details><summary>👀 What you should see</summary>

| T | RNN | LSTM | GRU |
|---|---|---|---|
| 10 | **1.00** | 1.00 | 1.00 |
| 30 | **0.51** | 1.00 | 1.00 |
| 60 | **0.50** | 1.00 | 1.00 |

- the plain RNN only solves short sequences - from T = 30 on it guesses: the gradient from the answer vanishes
  before it reaches the first bit (code break 3 of block 1)
- LSTM and GRU solve all three: their memory is a conveyor belt that is only multiplied by a gate close to 1
- each run takes a few seconds on a CPU (up to about 15 s)
- one detail (in the helper): we start the "keep" gate at "remember" (bias 3). Without this trick the LSTM also
  needs much longer at T = 60 - try `self.rnn.bias_hh_l0[...] = 0.0` in the helper cell if you are curious.
</details>

## 💻 Code break 3 - Bidirectional and stacked
Two more switches of the same layer: read the sequence **in both directions**, and put **several layers** on top of
each other.

In [25]:
BIDIRECTIONAL = True    # 👈 True
LAYERS = 1               # 👈 2


layer = nn.GRU(input_size=1, hidden_size=16, batch_first=True, bidirectional=BIDIRECTIONAL, num_layers=LAYERS)
out, _ = layer(beats)
print("out:", tuple(out.shape), "  parameters:", sum(p.numel() for p in layer.parameters()))

out: (8, 140, 32)   parameters: 1824


🔧 **Change it:** `BIDIRECTIONAL = True`. Then `BIDIRECTIONAL = False` and `LAYERS = 2`.

<details><summary>👀 What you should see</summary>

- normal GRU: out **(8, 140, 16)**, **912** parameters
- bidirectional: out **(8, 140, 32)** - 16 numbers from the forward GRU + 16 from the backward GRU at every step;
  **1 824** parameters (2 ×). The next Linear layer must expect 32 inputs.
- 2 layers: out **(8, 140, 16)** (only the top layer), **2 544** parameters - the second GRU reads the 16 memory
  numbers of the first one instead of the 1 ECG number, so it has more input weights
- bidirectional needs the **whole** sequence before it can answer - fine for one finished heartbeat, not for an
  alarm that must come while the patient is being recorded
</details>